<h1 style="color:#2E86AB; font-family:Georgia; text-align:center; border-bottom: 3px solid #2E86AB; padding-bottom:10px;">
📄 Extracción de Indicadores Educativos desde PDFs
</h1>

**Autor:** Eddy Luis  
**Fecha:** Mayo 2026  
**Herramientas:** Python · Pandas · pdfplumber

---

## 📌 Descripción
Extraemos las tablas de **promoción, reprobación y abandono** por Regional y Distrito
de los PDFs del MINERD (2021-2024), y las consolidamos en un solo DataFrame
para cruzar con el CSV de centros educativos.

---

## 📋 Contenido de este Notebook

1. ⚙️ Configuración
2. 🔧 Función de Extracción
3. 📥 Extracción de los 4 PDFs (2021-2024)
4. 🏷️ Separar Regional y Distrito
5. 🔄 Transformar a Formato Largo
6. 📊 Tabla Pivoteada de Abandono
7. 📈 Visualización
8. 💾 Guardar Datos Procesados
9. 🔗 Preview del Cruce con Centros Educativos

---

In [ ]:
import pandas as pd
import pdfplumber
import re
from pathlib import Path

<h2 style="color:#A23B72; font-family:Georgia;">
⚙️ 1. Configuración
</h2>

Páginas donde se encuentran las tablas de "Porcentaje de promoción, reprobados y abandono por nivel, según regional y distrito" en cada PDF.

In [ ]:
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

PDF_CONFIG = {
    2021: {"file": "indicadores educativos 2021.pdf", "pages": list(range(47, 53)), "lectivo": "2021-2022"},
    2022: {"file": "indicadores educativos 2022.pdf", "pages": list(range(41, 47)), "lectivo": "2022-2023"},
    2023: {"file": "indicadores educativos 2023.pdf", "pages": list(range(46, 52)), "lectivo": "2023-2024"},
    2024: {"file": "indicadores educativos 2024.pdf", "pages": list(range(47, 53)), "lectivo": "2024-2025"},
}

COLUMNAS_INDICADOR = [
    "abandono_inicial", "promovido_inicial", "reprobado_inicial",
    "abandono_primario", "promovido_primario", "reprobado_primario",
    "abandono_secundario", "promovido_secundario", "reprobado_secundario",
]

<h2 style="color:#A23B72; font-family:Georgia;">
🔧 2. Función de Extracción
</h2>

Cada línea de datos tiene el formato: `NOMBRE  1.5  98.5  0.0  2.5  94.1  3.4  5.7  88.1  6.2` (9 valores numéricos: Abandono/Promovido/Reprobado × Inicial/Primario/Secundario).

In [ ]:
PATTERN_9 = re.compile(
    r"^(.+?)\s+"
    r"([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+"
    r"([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+"
    r"([\d.]+)\s+([\d.]+)\s+([\d.]+)\s*$"
)

PATTERN_8 = re.compile(
    r"^(.+?)\s+"
    r"([\d.]+)\s+([\d.]+)\s+"
    r"([\d.]+)\s+([\d.]+)\s+([\d.]+)\s+"
    r"([\d.]+)\s+([\d.]+)\s+([\d.]+)\s*$"
)

SKIP_KEYWORDS = ["Anuario", "INICIAL", "Abandono", "Promovido", "Reprobado", "REGIONAL", "Todos los sectores"]


def extraer_indicadores_pdf(filepath, page_indices, año_lectivo):
    """Extrae filas de indicadores de las páginas indicadas de un PDF."""
    rows = []
    with pdfplumber.open(filepath) as pdf:
        for pi in page_indices:
            text = pdf.pages[pi].extract_text()
            if not text:
                continue
            for line in text.split("\n"):
                line = line.strip()
                if not line or any(kw in line for kw in SKIP_KEYWORDS):
                    continue

                match9 = PATTERN_9.match(line)
                if match9:
                    nombre = match9.group(1).strip()
                    vals = [float(match9.group(i)) for i in range(2, 11)]
                    rows.append([nombre, año_lectivo] + vals)
                    continue

                match8 = PATTERN_8.match(line)
                if match8:
                    nombre = match8.group(1).strip()
                    v = [float(match8.group(i)) for i in range(2, 10)]
                    # 2021-2022: sin Reprobado Inicial → insertar 0.0
                    vals = [v[0], v[1], 0.0, v[2], v[3], v[4], v[5], v[6], v[7]]
                    rows.append([nombre, año_lectivo] + vals)

    df = pd.DataFrame(rows, columns=["regional_distrito", "año_lectivo"] + COLUMNAS_INDICADOR)
    return df

<h2 style="color:#A23B72; font-family:Georgia;">
📥 3. Extracción de los 4 PDFs (2021-2024)
</h2>

In [ ]:
dfs = []
for year, cfg in PDF_CONFIG.items():
    filepath = RAW_DIR / cfg["file"]
    df_year = extraer_indicadores_pdf(filepath, cfg["pages"], cfg["lectivo"])
    print(f"{cfg['lectivo']}: {len(df_year)} filas extraídas")
    dfs.append(df_year)

df_indicadores = pd.concat(dfs, ignore_index=True)
print(f"\nTotal consolidado: {df_indicadores.shape}")
df_indicadores.head(10)

<h2 style="color:#A23B72; font-family:Georgia;">
🏷️ 4. Separar Regional y Distrito
</h2>

- **Regional**: código de 2 dígitos (ej. `01 - BARAHONA`)
- **Distrito**: código de 4 dígitos (ej. `0101 - PEDERNALES`)
- `TOTAL`: fila resumen nacional

In [ ]:
def clasificar_nivel(nombre):
    if nombre == "TOTAL":
        return "total", None, None
    code = nombre.split(" - ")[0].strip()
    if len(code) == 2 and code.isdigit():
        return "regional", nombre, None
    elif len(code) == 4 and code.isdigit():
        regional_code = code[:2]
        return "distrito", None, nombre
    return "otro", None, None

df_indicadores[["tipo", "regional", "distrito"]] = df_indicadores["regional_distrito"].apply(
    lambda x: pd.Series(clasificar_nivel(x))
)

# Propagar la regional hacia abajo para asociar cada distrito con su regional
current_regional = None
regionales = []
for _, row in df_indicadores.iterrows():
    if row["tipo"] == "regional":
        current_regional = row["regional_distrito"]
    elif row["tipo"] == "total":
        current_regional = None
    regionales.append(current_regional)

df_indicadores["regional"] = regionales

print(df_indicadores["tipo"].value_counts())
df_indicadores.head(15)